# NeSy trên nền Neuron

Code: `zeroshot/Neuron/model/nesy.py` (mô hình), `kg.py` (đọc KG), `main_nesy.py` (train / eval), config trong `config/nesy/`.

- **Nhánh neural**: head của Neuron (micro-prototype spatial + temporal), giữ nguyên, train bằng loss của Neuron.
- **Nhánh symbolic**: 82 slot primitive có tên (KG v2: `primitive_v2.json`, `kg_rules_v2.json`) (query lấy từ CLIP text của primitive, attention chỉ trên các khớp của primitive, tách bên trái / phải) → giá trị chân lý `p_k(t)` → luật require / forbid / sequence → `s_sym`.
- **Giám sát grounding trực tiếp** (`primitive_labels.py`): nhãn của từng primitive tính từ `definition` hình học trên skeleton (cùng crop với cube, không xoay), BCE với `p_k(t)` (`lambda_prim`). `ground_detach: True`: loss theo class không train grounding, grounding chỉ học từ nhãn primitive.
- **Fusion**: `z = τ·s_neu + β·s_sym` (β học được); lúc test loại các lớp mâu thuẫn số người.

Cột kết quả (mỗi mẫu **một** dự đoán): `zsl_s_neu` (Neuron, cộng spatial + temporal), `zsl_s_sym`, `zsl_z` (fused), `zsl_final` (fused + mask số người). `zsl_neuron_or` là cách tính của paper Neuron (đúng nếu spatial HOẶC temporal đúng), chỉ để đối chiếu. GZSL: γ dò trên test như Neuron.

`zsl_oracle`: luật chạy trên nhãn primitive của tập test (trần của KG, không học). `prim_auc_unseen` / `prim_auc_seen`: AUC của grounding so với nhãn primitive trên mẫu test của lớp unseen / seen (từng primitive: `prim_auc.json` trong thư mục run).

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, shutil, subprocess
import pandas as pd

DRIVE_DIR   = '/content/drive/MyDrive/BaoHuy/zeroshot'
WEIGHTS_DIR = f'{DRIVE_DIR}/shiftgcn_weights'
SKEL = {'ntu60':  '/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU60_CS.npz',
        'ntu120': '/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU120_CSub.npz'}
OUT_DIR = f'{DRIVE_DIR}/nesy_neuron'          # work_dir trong config/nesy/*.yaml

REPO = '/content/thesis'
if not os.path.exists(REPO):
    try:
        from google.colab import userdata
        tok = userdata.get('GH_TOKEN')
    except Exception:
        tok = None
    url = f'https://{tok}@github.com/NguyenBaoHuy05/thesis.git' if tok else 'https://github.com/NguyenBaoHuy05/thesis.git'
    subprocess.run(['git', 'clone', '-q', url, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
!git -C {REPO} log -1 --format='%h %s'
NEURON = f'{REPO}/zeroshot/Neuron'
print('weights:', len(glob.glob(f'{WEIGHTS_DIR}/**/*.pt', recursive=True)), 'file')

In [ ]:
# Giải nén npz trên Drive thẳng thành .npy trên đĩa local (đọc tuần tự, không nạp vào RAM).
# Feeder_Mmap đọc các .npy này bằng memmap, nên chạy được cả trên T4 (RAM ~12 GB).
import zipfile
def local_npy(ds):
    out = f"/content/data/{os.path.basename(SKEL[ds])[:-4]}"
    os.makedirs(out, exist_ok=True)
    with zipfile.ZipFile(SKEL[ds]) as zf:
        for name in zf.namelist():
            dst = f"{out}/{name}"
            if not os.path.exists(dst):
                with zf.open(name) as s, open(dst + ".tmp", "wb") as d:
                    shutil.copyfileobj(s, d, 64 << 20)
                os.replace(dst + ".tmp", dst)
    print(out, sorted(os.listdir(out)))

local_npy("ntu60")
!free -h | head -2; df -h /content | tail -1

In [ ]:
# Bước 0 (không train, vài phút): nhãn primitive tính từ definition có hợp lý không, và luật chạy trên chúng
# được bao nhiêu (ZSL oracle). In: tỷ lệ bật của từng primitive, primitive không bao giờ / gần như luôn bật,
# độ khớp với luật (require nên bật, forbid nên tắt), accuracy của luật trên train (lớp seen) và test (lớp unseen).
CFG = "ntu60_xsub_55_5"   # ntu60_xsub_55_5 | ntu60_xsub_48_12 | ntu120_xsub_110_10 | ntu120_xsub_96_24
if CFG.startswith("ntu120"):
    local_npy("ntu120")
!cd {NEURON} && python primitive_labels.py --config config/nesy/{CFG}.yaml --n 3000

In [ ]:
# Chạy một split. Lần đầu: mã hóa text primitive bằng CLIP, rồi trích cube của encoder một lần
# (mỗi mẫu train một augment cố định). cache_device auto tự chọn GPU / RAM / đĩa theo bộ nhớ còn trống.
# Bị ngắt thì chạy lại cell này: tiếp tục từ last.pt trong OUT_DIR/RUN/<split>.
CFG   = "ntu60_xsub_55_5"   # ntu60_xsub_55_5 | ntu60_xsub_48_12 | ntu120_xsub_110_10 | ntu120_xsub_96_24
RUN   = "v2_kg"             # thư mục con của nesy_neuron/: đổi tên mỗi khi thử cấu hình khác (v1 = KG cũ, v2_kg = KG v2)
EXTRA = ""                  # ghi đè config, vd "--cache-views 2" hoặc "--model-args neuron_scale_learn=True"
SPLIT = {"ntu60_xsub_55_5": "ntu60_seen55_unseen5", "ntu60_xsub_48_12": "ntu60_seen48_unseen12",
         "ntu120_xsub_110_10": "ntu120_seen110_unseen10", "ntu120_xsub_96_24": "ntu120_seen96_unseen24"}[CFG]
if CFG.startswith("ntu120"):
    local_npy("ntu120")
!cd {NEURON} && python main_nesy.py --config config/nesy/{CFG}.yaml --work-dir {OUT_DIR}/{RUN}/{SPLIT} {EXTRA}

In [ ]:
# Tổng hợp mọi run đã xong (run = thư mục con; v0 là run đầu tiên, nằm thẳng trong nesy_neuron/<split>)
rows = []
for f in sorted(glob.glob(f"{OUT_DIR}/**/summary.json", recursive=True)):
    s = json.load(open(f))
    run = os.path.relpath(os.path.dirname(os.path.dirname(f)), OUT_DIR)
    run = "v0" if run == "." else run
    last = s["last_epoch"]
    tail = pd.read_csv(os.path.join(os.path.dirname(f), "metrics.csv")).tail(10)   # trung bình 10 epoch cuối
    for name, r in (("last", last), ("mean last 10", tail.mean().to_dict()),
                    (f"best.pt (ep {s['best_epoch']})", s["best_checkpoint"])):
        rows.append(dict(run=run, split=s["split"], row=name,
                         **{k: r.get(k) for k in ("zsl_s_neu", "zsl_neu3", "zsl_s_sym", "zsl_z", "zsl_final",
                                                  "zsl_neuron_or", "gzsl_s_neu_H", "gzsl_final_H")}))
display(pd.DataFrame(rows).round(4))

In [ ]:
# Đường học theo epoch của CFG / RUN (đọc metrics.csv, xem được cả khi đang train)
path = f"{OUT_DIR}/{RUN}/{SPLIT}/metrics.csv"
if not os.path.exists(path):
    print("chưa có epoch nào xong:", path)
else:
    m = pd.read_csv(path)
    cols = [c for c in ("zsl_s_neu", "zsl_neu3", "zsl_s_sym", "zsl_z", "zsl_final", "zsl_neuron_or") if c in m]
    m.plot(x="epoch", y=cols, figsize=(9, 4), grid=True, title=f"ZSL theo epoch ({RUN} / {SPLIT})")
    show = ["epoch"] + cols + [c for c in ("gzsl_final_H", "beta", "neuron_scale", "L_neuron", "L_sym", "L_fuse",
                                           "L_logic", "time_s") if c in m]
    display(m[show].round(4))